# Fingerprint Descriptors + MLP Embedding

##### - MACCS
##### - Pharmacophore ErG
##### - PubChem
##### - ECFP

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random

from rdkit import Chem
from rdkit.Chem import AllChem
from torch.utils.data import Dataset, DataLoader

# ------------------------------
# 0) Set deterministic seed
# ------------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ------------------------------
# 1) Fingerprint function
# ------------------------------
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
import numpy as np
import torch

# สมมติว่าคุณมีฟังก์ชันนี้
def GetPubChemFPs(mol):
    """
    ตัวอย่างฟังก์ชัน PubChem fingerprint
    คืนค่าเป็น RDKit ExplicitBitVect
    """
    from rdkit.Chem import rdMolDescriptors
    return rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=881)

def extract_fingerprints(smiles: str) -> torch.Tensor:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"SMILES ไม่ถูกต้อง: {smiles}")

    # MACCS (167 bits)
    fp_maccs_rd = AllChem.GetMACCSKeysFingerprint(mol)
    fp_maccs = np.zeros((fp_maccs_rd.GetNumBits(),), dtype=int)
    DataStructs.ConvertToNumpyArray(fp_maccs_rd, fp_maccs)

    # Pharmacophore ErG (441 bits, numpy array)
    fp_erg_raw = AllChem.GetErGFingerprint(mol, fuzzIncrement=0.3, maxPath=21, minPath=1)
    fp_erg = np.array(fp_erg_raw, dtype=int)

    # PubChem fingerprint (881 bits)
    fp_pub_rd = GetPubChemFPs(mol)
    fp_pub = np.zeros((fp_pub_rd.GetNumBits(),), dtype=int)
    DataStructs.ConvertToNumpyArray(fp_pub_rd, fp_pub)

    # ECFP2 / Morgan fingerprint (1024 bits)
    fp_ecfp2_rd = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
    fp_ecfp2 = np.zeros((fp_ecfp2_rd.GetNumBits(),), dtype=int)
    DataStructs.ConvertToNumpyArray(fp_ecfp2_rd, fp_ecfp2)

    # Concatenate all fingerprints
    fingerprint = np.concatenate([fp_maccs, fp_erg, fp_pub, fp_ecfp2])

    # แปลงเป็น PyTorch tensor
    return torch.tensor(fingerprint, dtype=torch.float32)

# ------------------------------
# 2) Dataset
# ------------------------------
class SMILESDataset(Dataset):
    def __init__(self, smiles_list, targets):
        self.smiles_list = smiles_list
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, idx):
        fp = extract_fingerprints(self.smiles_list[idx])
        return fp, self.targets[idx]

# ------------------------------
# 3) MLP
# ------------------------------
class FingerprintMLP(nn.Module):
    def __init__(self, input_dim, output_dim=64, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        x = self.norm(x)
        x = self.dropout(x)
        return x

# ------------------------------
# 4) Train function
# ------------------------------
def train_mlp(smiles_list, target_list, mlp_output_dim=64, epochs=50, batch_size=32, lr=1e-3):

    dataset = SMILESDataset(smiles_list, target_list)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    input_dim = extract_fingerprints(smiles_list[0]).shape[0]

    mlp = FingerprintMLP(input_dim=input_dim, output_dim=mlp_output_dim)
    optimizer = optim.Adam(mlp.parameters(), lr=lr)
    criterion = nn.MSELoss()

    mlp.train()
    for epoch in range(epochs):
        total_loss = 0
        for fp, tgt in loader:
            optimizer.zero_grad()
            output = mlp(fp)
            if tgt.ndim == 1:
                tgt = tgt.unsqueeze(1)
            loss = criterion(output, tgt.expand_as(output))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(loader):.4f}")

    return mlp

# ------------------------------
# 5) Generate feature (train/test separated)
# ------------------------------
def generate_features_from_model(df, mlp, output_csv):
    smiles_list = df["Smiles"].astype(str).tolist()

    # ถ้ามี target ในไฟล์ test จะเก็บไว้ด้วย (ไม่ฝึก)
    if len(df.columns) > 1:
        target_list = df[df.columns[-1]].tolist()
    else:
        target_list = [None] * len(smiles_list)

    mlp.eval()
    vectors = []
    valid_smiles = []
    valid_targets = []

    for smi, tgt in zip(smiles_list, target_list):
        try:
            fp = extract_fingerprints(smi)
            vec = mlp(fp).detach().numpy()
            vectors.append(vec)
            valid_smiles.append(smi)
            valid_targets.append(tgt)
        except Exception as e:
            print(f"❌ ข้าม SMILES: {smi} | error: {e}")

    vec_df = pd.DataFrame(vectors)
    vec_df.insert(0, "smiles", valid_smiles)
    vec_df["Target"] = valid_targets
    vec_df.to_csv(output_csv, index=False)
    print(f"✓ Saved → {output_csv}")


# ------------------------------
# 6) Main: Train on train.xlsx → Embed test.xlsx
# ------------------------------
def run_pipeline(train_path, test_path,
                 out_train_csv="supervised_alt_train.csv",
                 out_test_csv="supervised_alt_test.csv",
                 mlp_output_dim=64, epochs=50):

    # --- Load Train ---
    df_train = pd.read_csv(train_path)
    train_smiles = df_train["Smiles"].astype(str).tolist()
    #train_targets = df_train[df_train.columns[-1]].tolist()
    train_targets = df_train["Class"].astype(float).tolist()

    # --- Train MLP ---
    mlp = train_mlp(train_smiles, train_targets,
                    mlp_output_dim=mlp_output_dim,
                    epochs=epochs)

    # --- Generate Train Embedding ---
    generate_features_from_model(df_train, mlp, out_train_csv)

    # --- Load Test ---
    df_test = pd.read_csv(test_path)

    # --- Generate Test Embedding (no training) ---
    generate_features_from_model(df_test, mlp, out_test_csv)


# ------------------------------
# 7) Run
# ------------------------------
run_pipeline(
    train_path="/kaggle/input/datasets/paxpoom/nai-classification-raw/NAI2_Train.csv",
    test_path="/kaggle/input/datasets/paxpoom/nai-classification-raw/NAI2_Test.csv",
    out_train_csv="supervised_NAI_train.csv",
    out_test_csv="supervised_NAI_test.csv",
    mlp_output_dim=64,
    epochs=50
)


In [2]:
!pip install rdkit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 45.8 MB/s eta 0:00:00:00:0100:01
